In [ ]:
#!pip -q install "transformers==4.45.2" datasets evaluate scikit-learn imbalanced-learn \
 #                nltk emoji matplotlib seaborn wordcloud pandas openpyxl tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install pandas openpyxl tqdm emoji

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import emoji

SEED = 59
random.seed(SEED)
np.random.seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# PATHS
DATA_PATH       = "/content/drive/MyDrive/TA ABSA/data_kaggle/shopee_ulasan_kaggle.xlsx"
STOPWORDS_PATH  = "/content/drive/MyDrive/TA ABSA/stopwords.txt"
SLANGWORDS_PATH = "/content/drive/MyDrive/TA ABSA/slangwords.txt"
OUT_DIR         = "/content/drive/MyDrive/TA ABSA/outputs_absa_stage1_v2"
os.makedirs(OUT_DIR, exist_ok=True)


# COLUMNS
ID_COL   = "review_id"
TEXT_COL = "content"

In [ ]:
# LOAD SLANGWORDS
slangwords = {}
with open(SLANGWORDS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(",", 1)
        if len(parts) == 2:
            old = parts[0].strip().lower()
            new = parts[1].strip().lower()
            if old:
                slangwords[old] = new

print("Jumlah slangwords:", len(slangwords))

Jumlah slangwords: 15947


In [ ]:
# CLEANSING FUNCTIONS

def remove_emoji(text: str) -> str:
    return emoji.replace_emoji(str(text), replace=" ")

def convertToSlangword(text: str) -> str:
    toks = str(text).split()
    out = []
    for kata in toks:
        kata = kata.lower()
        out.append(slangwords.get(kata, kata))
    return " ".join(out)

def replaceThreeOrMore(text: str) -> str:
    # hanya huruf, angka aman
    pattern = re.compile(r"([a-zA-Z])\1{2,}", flags=re.UNICODE)
    return pattern.sub(r"\1", text)

def filtering(text: str, remove_numbers: bool=False) -> str:
    text = str(text)

    # remove emoji
    text = remove_emoji(text)

    # remove unicode escape
    text = re.sub(r"(\\u[0-9A-Fa-f]{4,6})", " ", text)

    # remove url
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove mention
    text = re.sub(r"@\w+\b", " ", text)

    # remove hashtag symbol, keep word
    text = re.sub(r"#(\w+)", r"\1", text)

    # optional remove numbers
    if remove_numbers:
        text = re.sub(r"\d+", " ", text)

    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

def clean(text: str, slang: bool=True, num: bool=False) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    text = text.lower()
    text = filtering(text, remove_numbers=num)
    text = replaceThreeOrMore(text)

    if slang:
        text = convertToSlangword(text)

    return text.strip()


In [ ]:
# LOAD  & CREATE UNIQUE ID

df = pd.read_excel(DATA_PATH)

# remove BOM dari nama kolom
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

# SOLUSI: Karena tidak ada ID, kita buat ID unik berdasarkan urutan baris
print("Kolom ID tidak ditemukan. Membuat review_id otomatis...")
df[ID_COL] = range(1, len(df) + 1)

print("Shape awal:", df.shape)
print("Kolom:", list(df.columns))

assert TEXT_COL in df.columns, f"Kolom wajib '{TEXT_COL}' tidak ditemukan!"

Kolom ID tidak ditemukan. Membuat review_id otomatis...
Shape awal: (2937, 40)
Kolom: ['review_id', 'user', 'content', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39']


In [ ]:
# DROP NA & DEDUP
df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)

if ID_COL in df.columns and df[ID_COL].notna().any():
    df = df.drop_duplicates(subset=[ID_COL]).copy()
else:
    df = df.drop_duplicates(subset=[TEXT_COL]).copy()

print("Shape setelah dedup & drop NA:", df.shape)


Shape setelah dedup & drop NA: (2937, 40)


In [ ]:
# CLEANING
tqdm.pandas(desc="Cleaning text")
df["clean"] = df[TEXT_COL].progress_apply(lambda x: clean(x, slang=True, num=False))


df = df[df["clean"].str.strip() != ""].reset_index(drop=True)


df_out = df[[ID_COL,TEXT_COL, "clean"]].copy()

print("Final shape:", df_out.shape)
display(df_out.head(10))

Cleaning text:   0%|          | 0/2937 [00:00<?, ?it/s]

Final shape: (2937, 3)


,review_id,content,clean
0,1,"Banyak membantu..dalam jual beli, produk berku...",banyak bantu dalam jual beli produk kualitas h...
1,2,Ayo teman-teman kita mendownload shopee Karena...,ayo teman teman kita mendownload shopee karena...
2,3,"App sangggatttt rekomendasiiiiiii, buat belanj...",app sangat rekomendasi buat belanja semakin ke...
3,4,"Shopee memang is the bast, tapi kalu boleh sar...",shopee memang is the bast tapi kalau boleh sar...
4,5,Aplikasi belanja mudah segala sesuatu kebutuha...,aplikasi belanja mudah segala sesuatu kebutuha...
5,6,Aku kasih bintang 5 deh.. mudah.mudahan cepet ...,aku kasih bintang 5 deh mudah mudahan cepat da...
6,7,Baabi emng aplikasi ini. Penipuan. Setiap saya...,baabi memang aplikasi ini penipuan tiap saya p...
7,8,Ayo donk shopee buat aplikasi car sama bike ju...,ayo dong shopee buat aplikasi car sama bike ju...
8,9,"Suka banget ma shopee the best banget deh ,, s...",suka banget ma shopee the best banget deh sela...
9,10,Pengalaman aku slama blanja dishopee slalu pua...,pengalaman aku lama blanja dishopee selalu pua...


In [ ]:

out_csv  = os.path.join(OUT_DIR, "stage1_V2_content_clean.csv")
out_xlsx = os.path.join(OUT_DIR, "stage1_V2_content_clean.xlsx")


df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")


df_out.to_excel(out_xlsx, index=False)

print("File berhasil disimpan:")
print("CSV  :", out_csv)
print("XLSX :", out_xlsx)

File berhasil disimpan:
CSV  : /content/drive/MyDrive/TA ABSA/outputs_absa_stage1_v2/stage1_V2_content_clean.csv
XLSX : /content/drive/MyDrive/TA ABSA/outputs_absa_stage1_v2/stage1_V2_content_clean.xlsx
